In [1]:
import sys
sys.path.append("..")

import jax, os, corner
import jax.numpy as jnp
from jax import grad, config
import matplotlib.pyplot as plt
import numpy as np
config.update("jax_enable_x64", True)
config.update("jax_debug_nans", True)
from functools import partial
from models.priors import Mc_eta_uniform_masses_draw


# Load reparameterization methods
# from src.reparameterization import sigma, logistic_CDF, reparameterized_gradient

# Load birth/death method
# from src.birth_death import birth_death
from src.pampel import ula_sampler_full_jax_jit
from src.helper import rejection_sampling

print(jax.devices())

[cuda(id=0)]


In [ ]:
# def scan(f, init, xs, length=None):
#   """ 
#   - Note that lax.scan automatically JIT compiles
#   - Replace with this method instead as a hack for standard debugging
#   """
#   if xs is None:
#     xs = [None] * length
#   carry = init
#   ys = []
#   for x in xs:
#     carry, y = f(carry, x)
#     ys.append(y)
#   return carry, np.stack(ys)

# Mirrored Langevin Birth/Death (MLBD)

In [ ]:
# def ula_kernel(key, X, potential, grad_potential, dt, iteration, lower, upper, stride, rate, bandwidth, bounded_coordinates, periodic_coordinates):
#     """ 
#     Remarks
#     -------
#     (1) A subkey is immediately used. The key is used to split
#     (2) The periodic coordinates must begin at 0 for the modding to work nicely!!! Otherwise a shift in coordinates in necessary
#     """
#     N = X.shape[0]

#     # Calculate gradients
#     gmlpt_X = grad_potential(X)

#     # Update bounded coordinates
#     Y, gmlpt_Y = reparameterized_gradient(X[:, bounded_coordinates], gmlpt_X[:, bounded_coordinates], lower[bounded_coordinates], upper[bounded_coordinates])
#     key, subkey = jax.random.split(key)
#     Y = Y - gmlpt_Y * dt #+ jnp.sqrt(2 * dt) * jax.random.normal(key=subkey, shape=(N, len(bounded_coordinates)))
#     X = X.at[:, bounded_coordinates].set(sigma(logistic_CDF(Y), lower[bounded_coordinates], upper[bounded_coordinates]))

#     # Update periodic coordinates
#     key, subkey = jax.random.split(key)    
#     X = X.at[:, periodic_coordinates].add(-gmlpt_X[:, periodic_coordinates] * dt) #+ jnp.sqrt(2 * dt) * jax.random.normal(key=subkey, shape=(N, len(periodic_coordinates))))
#     X = X.at[:, periodic_coordinates].set(jnp.mod(X[:, periodic_coordinates], upper[periodic_coordinates])) 

#     # Perform jumps in primal space
#     key, subkey = jax.random.split(key)
#     # jumps = jax.lax.cond(jnp.mod(iteration, stride) == 0, lambda: birth_death(subkey, X, potential, bandwidth=bandwidth, rate=rate, a=lower, b=upper, bounded_coordinates=bounded_coordinates, periodic_coordinates=periodic_coordinates, sigma=standard_dev, f=f), lambda: jnp.arange(N))
#     # X = X[jumps]

#     iteration = iteration + 1
#     return key, X, iteration

# @partial(jax.jit, static_argnums=(1,2,3))
# def ula_sampler_full_jax_jit(key, potential, grad_potential, n_iter, dt, x_0, lower, upper, stride, rate, bandwidth, b_idx, p_idx):

#     # @progress_bar_scan(n_iter)
#     # @scan_tqdm(1000)
#     # @scan_tqdm(n_iter, print_rate=1, desc='progress bar', position=0, leave=False)
#     def ula_step(carry, x):
#         key, param, iteration = carry
#         key, param, iteration = ula_kernel(key, param, potential, grad_potential, dt, iteration, lower, upper, stride, rate, bandwidth, b_idx, p_idx)
#         return (key, param, iteration), param

#     carry = (key, x_0, 0)
#     _, samples = jax.lax.scan(ula_step, carry, None, n_iter)
#     # _, samples = scan(ula_step, carry, None, n_iter)
#     return samples

# Unit tests

In [ ]:
# def rejection_sampling(iid_samples, lower_bound, upper_bound):
#     truth_table = ((iid_samples > lower_bound) & (iid_samples < upper_bound))
#     idx = np.where(np.all(truth_table, axis=1))[0]
#     print('%i samples obtained from rejection sampling' % idx.shape[0])
#     return np.array(iid_samples[idx])

In [ ]:
""" 
Remarks
-------
(1) 2d MoG with increasing weights to the right
(2) Sampler teleports out all particles from smallest mode given enough time
(3) Can be mitigated using a small `rate`

"""

from models.mog_new import MoG 

k = 3
d = 2
weights = jnp.array([2, 4, 5])

mus = jnp.zeros((k, d))
mus = mus.at[0].set(jnp.array([-15, 0]))
# mus = mus.at[0].set(jnp.array([-10, 0]))
mus = mus.at[1].set(jnp.array([0, 0]))
# mus = mus.at[2].set(jnp.array([10, 0]))
mus = mus.at[2].set(jnp.array([15, 0]))

covs = jnp.zeros((k, d))
covs = covs.at[0].set(jnp.ones(d))
covs = covs.at[1].set(jnp.ones(d))
covs = covs.at[2].set(jnp.ones(d))

lower_bound = jnp.array([-15, -15])
upper_bound = jnp.array([15, 15])

model = MoG(weights, mus, covs, lower_bound, upper_bound)

iid_samples = model.newDrawFromPosterior(1000000)

bounded_iid_samples = rejection_sampling(iid_samples, model.lower_bound, model.upper_bound)

In [ ]:
# OLD SETTINGS
# ------------
# n_iter = 20000
# n_particles = 500
# eps = 1e-3
# stride = 100
# rate = 0.01
# bandwidth = 0.001

# Setup and run sampler
n_iter = 1000
n_particles = 200
eps = 1e-3
stride = 10
rate = 0.1
bandwidth = jnp.ones(d) * 1
p = 2
X0 = model._newDrawFromPrior(n_particles)
key = jax.random.PRNGKey(0)
sam = ula_sampler_full_jax_jit(key, jax.vmap(model.potential), jax.vmap(jax.jacfwd(model.potential)), n_iter, eps, X0, model.lower_bound, model.upper_bound, stride, rate, bandwidth=bandwidth)
# sam = ula_sampler_full_jax_jit(key, jax.vmap(model.potential), jax.vmap(jax.jacfwd(model.potential)), n_iter, eps, sam[-1], model.lower_bound, model.upper_bound, stride, rate)

In [ ]:
# Plot
import matplotlib.lines as mlines
reshaped_matrix = np.array(sam.reshape((sam.shape[0] * sam.shape[1], sam.shape[2])))
fig = corner.corner(bounded_iid_samples[-20000:], hist_kwargs={'density':True}, truths=jnp.mean(bounded_iid_samples, axis=0), color='k') 
labels = [r'$x_1$', r'$x_2$']
k_line = mlines.Line2D([], [], color='k', label='Truth')
r_line = mlines.Line2D([], [], color='r', label='MLBD')
corner.corner(reshaped_matrix[-20000:], color='r', fig=fig, hist_kwargs={'density':True}, labels=labels)
plt.legend(handles=[k_line,r_line], bbox_to_anchor=(0., 1.0, 1., .0), loc=4)

In [ ]:
""" 
Remarks
-------
(1) 2d MoG with all modes of similar weight
"""

from models.mog_new import MoG 

k = 3
d = 2
weights = jnp.array([5, 4, 5])

mus = jnp.zeros((k, d))
mus = mus.at[0].set(jnp.array([-10, 0]))
mus = mus.at[1].set(jnp.array([0, 0]))
mus = mus.at[2].set(jnp.array([10, 0]))

covs = jnp.zeros((k, d))
covs = covs.at[0].set(jnp.ones(d))
covs = covs.at[1].set(jnp.ones(d))
covs = covs.at[2].set(jnp.ones(d))

lower_bound = jnp.array([-15, -15])
upper_bound = jnp.array([15, 15])

model = MoG(weights, mus, covs, lower_bound, upper_bound)


iid_samples = model.newDrawFromPosterior(1000000)

bounded_iid_samples = rejection_sampling(iid_samples, model.lower_bound, model.upper_bound)

In [ ]:
# Setup and run sampler
n_iter = 20000
n_particles = 200
eps = 1e-2
stride = 100
rate = 0.01
bandwidth = 0.1
p = 0.5
X0 = model._newDrawFromPrior(n_particles)
key = jax.random.PRNGKey(0)
sam = ula_sampler_full_jax_jit(key, jax.vmap(model.potential), jax.vmap(jax.jacfwd(model.potential)), n_iter, eps, X0, model.lower_bound, model.upper_bound, stride, rate)

In [ ]:
# Plot
import matplotlib.lines as mlines
reshaped_matrix = np.array(sam.reshape((sam.shape[0] * sam.shape[1], sam.shape[2])))
fig = corner.corner(bounded_iid_samples[-20000:], hist_kwargs={'density':True}, truths=jnp.mean(bounded_iid_samples, axis=0), color='k') 
labels = [r'$x_1$', r'$x_2$']
k_line = mlines.Line2D([], [], color='k', label='Truth')
r_line = mlines.Line2D([], [], color='r', label='MLBD')
corner.corner(reshaped_matrix[-20000:], color='r', fig=fig, hist_kwargs={'density':True}, labels=labels)
plt.legend(handles=[k_line,r_line], bbox_to_anchor=(0., 1.0, 1., .0), loc=4)

In [ ]:
""" 
Remarks
-------
(1) 15d MoG with all modes of similar weight
"""

from models.mog_new import MoG 

k = 3
d = 15
weights = jnp.array([5, 4, 5])

mus = jnp.zeros((k, d))

mus = mus.at[0].set(jnp.array([-10, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]))
mus = mus.at[1].set(jnp.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]))
mus = mus.at[2].set(jnp.array([10, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]))

covs = jnp.zeros((k, d))
covs = covs.at[0].set(jnp.ones(d))
covs = covs.at[1].set(jnp.ones(d))
covs = covs.at[2].set(jnp.ones(d))

lower_bound = jnp.ones(d) * -15 
upper_bound = jnp.ones(d) * 15 

model = MoG(weights, mus, covs, lower_bound, upper_bound)


iid_samples = model.newDrawFromPosterior(1000000)

bounded_iid_samples = rejection_sampling(iid_samples, model.lower_bound, model.upper_bound)

In [ ]:
# Setup and run sampler
n_iter = 20000
n_particles = 200
eps = 1e-3
stride = 100
rate = 0.01
bandwidth = 100
p = 2
X0 = model._newDrawFromPrior(n_particles)
key = jax.random.PRNGKey(1)
sam = ula_sampler_full_jax_jit(key, jax.vmap(model.potential), jax.vmap(jax.jacfwd(model.potential)), n_iter, eps, X0, model.lower_bound, model.upper_bound, stride, rate)

In [ ]:
# Plot
import matplotlib.lines as mlines
reshaped_matrix = np.array(sam.reshape((sam.shape[0] * sam.shape[1], sam.shape[2])))
fig = corner.corner(bounded_iid_samples[-20000:], hist_kwargs={'density':True}, truths=jnp.mean(bounded_iid_samples, axis=0), color='k') 
# labels = [r'$x_1$', r'$x_2$']
k_line = mlines.Line2D([], [], color='k', label='Truth')
r_line = mlines.Line2D([], [], color='r', label='MLBD')
corner.corner(reshaped_matrix[-20000:], color='r', fig=fig, hist_kwargs={'density':True})#, labels=labels)
plt.legend(handles=[k_line,r_line], bbox_to_anchor=(0., 1.0, 1., .0), loc=4)

In [ ]:
""" 
Remarks
-------
(1) 15d MoG with some mass at the boundary
"""

from models.mog_new import MoG 

k = 3
d = 15
weights = jnp.array([0.5, 1, 2])

mus = jnp.zeros((k, d))

mus = mus.at[0].set(jnp.array([-15, 0, 0, 9, 0, 0, 0, 5, 0, 0, 0, 12, 0, 7, 0]))
mus = mus.at[1].set(jnp.array([0, 0, 0, -15, 0, 0, 4, 0, 10, 0, 0, 15, 0, 8, 0]))
mus = mus.at[2].set(jnp.array([10, 0, 0, 0, 3, 0, 2, 0, 0, 0, 0, 0, 0, 0, 15]))

covs = jnp.zeros((k, d))
covs = covs.at[0].set(jnp.ones(d))
covs = covs.at[1].set(jnp.ones(d))
covs = covs.at[2].set(jnp.ones(d))

lower_bound = jnp.ones(d) * -15 
upper_bound = jnp.ones(d) * 15 

model = MoG(weights, mus, covs, lower_bound, upper_bound)


iid_samples = model.newDrawFromPosterior(1000000)

bounded_iid_samples = rejection_sampling(iid_samples, model.lower_bound, model.upper_bound)

In [ ]:
bandwidth

In [ ]:
# Setup and run sampler
n_iter = 50000
n_particles = 500
eps = 1e-2
stride = 100
# rate = 0.001
rate = 0.001
bandwidth = jnp.ones(d) * 1
p = 2
X0 = model._newDrawFromPrior(n_particles)
key = jax.random.PRNGKey(1)
sam = ula_sampler_full_jax_jit(key, jax.vmap(model.potential), jax.vmap(jax.jacfwd(model.potential)), n_iter, eps, X0, model.lower_bound, model.upper_bound, stride, rate, bandwidth=bandwidth)

In [ ]:
# Plot
import matplotlib.lines as mlines
reshaped_matrix = np.array(sam.reshape((sam.shape[0] * sam.shape[1], sam.shape[2])))
fig = corner.corner(bounded_iid_samples[-20000:], hist_kwargs={'density':True}, truths=jnp.mean(bounded_iid_samples, axis=0), color='k') 
# labels = [r'$x_1$', r'$x_2$']
k_line = mlines.Line2D([], [], color='k', label='Truth')
r_line = mlines.Line2D([], [], color='r', label='MLBD')
corner.corner(reshaped_matrix[-20000:], color='r', fig=fig, hist_kwargs={'density':True})#, labels=labels)
plt.legend(handles=[k_line,r_line], bbox_to_anchor=(0., 1.0, 1., .0), loc=4)

# GW150914

In [2]:
from models.gw150914 import gwfast_LVGW150914

# Initialize model
model = gwfast_LVGW150914(wf_model='IMRPhenomD', nbins=100, verbose=True)
# model = gwfast_LVGW150914(wf_model='TaylorF2', nbins=100, verbose=True)


Using waveform model: IMRPhenomD
fmin:  10.0
fmax:  560.0
df_standard:  5.5
nbins:  100
Using ASD from file /home/al44828/projects/sSVN_GW/notebooks/aLIGO_O4_high_asd.txt 
Using ASD from file /home/al44828/projects/sSVN_GW/notebooks/aLIGO_O4_high_asd.txt 
Using ASD from file /home/al44828/projects/sSVN_GW/notebooks/AdV_asd.txt 
SNR at true values: 91.57


In [ ]:
from models.priors import Mc_eta_uniform_masses_draw
import corner
import matplotlib.pyplot as plt

samples = Mc_eta_uniform_masses_draw(5000, jnp.array([32, 0.24]), jnp.array([40, 0.249]))

corner.corner(samples, labels=['Mc', 'eta'])

plt.scatter(samples[:,0], samples[:,1], s=0.1)


In [4]:
# Setup and run sampler
n_iter = 100
n_particles = 200
eps = 1e-6

# Birth death
stride = n_iter + 1 
rate = 1e-6 
bandwidth = jnp.ones(model.DoF) * 0.01
bandwidth = bandwidth.at[jnp.array([4, 6, 8])].set(jnp.ones(3) * 0.01)

# Initial draw from prior
X0 = model._newDrawFromPrior(n_particles)
X0 = X0.at[:, jnp.array([0, 1])].set(Mc_eta_uniform_masses_draw(n_particles, model.lower_bound[0:2], model.upper_bound[0:2]))

# We now have to change the Mc, eta samples to be uniformly drawn from m1, m2!
# np.random.seed(1)
# m = 1000000
# m1_sams = np.random.uniform(low=10, high=80, size=m)
# m2_sams = np.random.uniform(low=10, high=80, size=m)

# idxs = np.argwhere(m1_sams > m2_sams)
# m1_sams = m1_sams[idxs].squeeze()
# m2_sams = m2_sams[idxs].squeeze()

# Mc_sams = (m1_sams * m2_sams) ** (3 / 5) / ((m1_sams + m2_sams) ** (1/5))
# eta_sams = (m1_sams * m2_sams) / ((m1_sams + m2_sams) ** (2))
# eta_sams = eta_sams * model.eta_fudge_factor

# m = len(m1_sams)

# vec = np.zeros((m, 2))
# vec[:,0] = Mc_sams
# vec[:,1] = eta_sams
# new_sams = jnp.array(rejection_sampling(vec, model.lower_bound[0:2], model.upper_bound[0:2]))

# X0 = X0.at[:, jnp.array([0, 1])].set(new_sams[0:n_particles])

# Warm start
# X0 = jnp.tile(model.true_params, n_particles).reshape(n_particles, 11)
# X0 = X0.at[:,8].set(np.random.uniform(0, 2 * jnp.pi, size=n_particles))
# X0 = X0.at[:,6].set(np.random.uniform(0, jnp.pi, size=n_particles))

key = jax.random.PRNGKey(0)

sam = ula_sampler_full_jax_jit(key, model.minusLogLikelihood, model.gradient_minusLogLikelihood, n_iter, eps, X0, model.lower_bound, model.upper_bound, stride, rate, bandwidth, model.bounded_coordinates, model.periodic_coordinates)

buffer in prior: 0.000000


NameError: name 'Mc_eta_uniform_masses_draw' is not defined

In [ ]:
sam = ula_sampler_full_jax_jit(key, model.minusLogLikelihood, model.gradient_minusLogLikelihood, n_iter, eps, sam[-1], model.lower_bound, model.upper_bound, stride, rate, bandwidth, model.bounded_coordinates, model.periodic_coordinates)

In [ ]:
reshaped_matrix = np.array(sam.reshape((sam.shape[0] * sam.shape[1], 11)))
fig = corner.corner(reshaped_matrix[-50000:], hist_kwargs={'density':True}, labels=model.gwfast_param_order, truths=model.true_params)

In [ ]:
vals = model.gradient_minusLogLikelihood(sam[-1])

In [ ]:
jnp.norm

In [ ]:
variances = np.var(np.abs(vals), axis=0)

# Plot the gradient variance
for i in np.argsort(variances)[::-1]:
        plt.hist(vals[:,i], bins=50, label=model.gwfast_param_order[i])
        print(np.max(np.abs(vals[:,i])), i)
plt.title('Gradient variance')
plt.legend()

# Cross sections test

In [ ]:
priorDict = model.priorDict
injection = model.true_params
DoF = model.DoF
def getCrossSection(index1, index2, func, ngrid, DoF=DoF):
    # a, b are the parameters for which we want the marginals:
    param1 = model.gwfast_param_order[index1]
    param2 = model.gwfast_param_order[index2]
    x = np.linspace(priorDict[param1][0], priorDict[param1][1], ngrid)
    y = np.linspace(priorDict[param2][0], priorDict[param2][1], ngrid)

    # x = np.linspace(priorDict[index1][0], priorDict[index1][1], ngrid)
    # y = np.linspace(priorDict[index2][0], priorDict[index2][1], ngrid)
    X, Y = np.meshgrid(x, y)
    particle_grid = np.zeros((ngrid ** 2, DoF))
    parameter_mesh = np.vstack((np.ndarray.flatten(X), np.ndarray.flatten(Y))).T
    particle_grid[:, index1] = parameter_mesh[:, 0]
    particle_grid[:, index2] = parameter_mesh[:, 1]
    for i in range(DoF): # Fix all other parameters
        if i != index1 and i != index2:
            particle_grid[:, i] = np.ones(ngrid ** 2) * injection[i]
    Z = func(particle_grid).reshape(ngrid,ngrid)
    fig, ax = plt.subplots(figsize = (5, 5))
    cp = ax.contourf(X, Y, Z)
    ax.scatter(model.true_params[index1], model.true_params[index2], s=0.1, marker='x', c='r')
    ax.axis('equal')
    plt.colorbar(cp)
    ax.set_xlabel(model.gwfast_param_order[index1])
    ax.set_ylabel(model.gwfast_param_order[index2])
    ax.set_title('Likelihood cross section')
    filename = str(index1) + str(index2) + '.png'
    path = os.path.join('marginals', filename)
    # fig.savefig(path)

posterior = jax.jit(lambda X: jnp.exp(-1 * model.minusLogLikelihood(X)))

neg_potential = jax.jit(lambda X: -1 * model.minusLogLikelihood(X))
# neg_potential = lambda X: -1 * model.minusLogLikelihood(X)

# getCrossSection(0, 1, posterior, 100)
# getCrossSection(0, 1, neg_potential, 100)

for i in range(DoF):
# for i in [1]:
    for j in range(i+1, DoF):
    # for j in [0]:
        print('Getting cross section for %i, %i' % (i,j))
        # getCrossSection(i, j, posterior, 300)
        getCrossSection(i, j, neg_potential, 100)

# Subset unit tests

In [ ]:
n_particles = 2

# subset_params = jnp.array([0, 1, 2, 3, 5, 6, 7, 9, 10])
subset_params = jnp.array([0, 1, 2, 3, 4, 5, 6, 7])
lower = model.lower_bound[subset_params] 
upper = model.upper_bound[subset_params]

# injection wrapper
DoF = len(model.gwfast_param_order)
injection = np.zeros(DoF)
for d in range(DoF):
    injection[d] = model.injParams[model.gwfast_param_order[d]]

X_injection = np.tile(injection, n_particles).reshape(n_particles, DoF)
X_injection_gpu = jnp.array(X_injection)

# Initial draw from reduced prior
X0_all = model._newDrawFromPrior(n_particles)

# We now have to change the Mc, eta samples to be uniformly drawn from m1, m2!
np.random.seed(1)
m = 1000000
m1_sams = np.random.uniform(low=10, high=80, size=m)
m2_sams = np.random.uniform(low=10, high=80, size=m)

Mc_sams = (m1_sams * m2_sams) ** (3 / 5) / ((m1_sams + m2_sams) ** (1/5))
eta_sams = (m1_sams * m2_sams) / ((m1_sams + m2_sams) ** (2))

vec = np.zeros((m, 2))
vec[:,0] = Mc_sams
vec[:,1] = eta_sams
new_sams = jnp.array(rejection_sampling(vec, model.lower_bound[0:2], model.upper_bound[0:2]))

X0_all = X0_all.at[:, jnp.array([0, 1])].set(new_sams[0:n_particles])

X0_subset = X0_all[:, subset_params]

def potential_subset(X_red):
    X_ = X_injection_gpu.at[:, subset_params].set(X_red)
    return model.minusLogLikelihood(X_)

def gradient_subset(X_red):
    X_ = X_injection_gpu.at[:, subset_params].set(X_red)
    return model.gradient_minusLogLikelihood(X_)[:, subset_params]
    


In [ ]:
# FOR BIRTH DEATH
n_iter = 1000000
eps = 1e-7
stride = n_iter + 1 # Worked well with this! #n_iter + 1
rate = 1e-7 # FOR BIRTH DEATH
bandwidth = jnp.ones(len(subset_params)) * 100

key = jax.random.PRNGKey(0)

sam = ula_sampler_full_jax_jit(key, potential_subset, gradient_subset, n_iter, eps, X0_subset, lower, upper, stride, rate, bandwidth=bandwidth)

In [ ]:
sam = ula_sampler_full_jax_jit(key, potential_subset, gradient_subset, n_iter, eps, sam[-1], lower, upper, stride, rate, bandwidth=bandwidth)

In [ ]:
reshaped_matrix = np.array(sam.reshape((sam.shape[0] * sam.shape[1], len(subset_params))))
fig = corner.corner(reshaped_matrix[-100000:], hist_kwargs={'density':True}, labels=np.array(model.gwfast_param_order)[subset_params], truths=model.true_params[subset_params])

In [ ]:
import matplotlib.pyplot as plt
n_iter = 100000
plt.plot(np.arange(n_iter), gamma(np.arange(n_iter), T=n_iter, c=3, p=5))

In [ ]:
from jax_tqdm import scan_tqdm

In [ ]:
# Settings for run
n_iter = 100000

# n_iter = 100
n_particles = 200
eps = 1e-6

X0 = model._newDrawFromPrior(n_particles)
key = jax.random.PRNGKey(0)
# eps = jnp.ones(11) * 5e-7
# eps = eps.at[1].set(5e-5)
# eps = eps.at[4].set(1e-4)
# eps = eps.at[8].set(1e-4)

# For mixture of Gaussian model
# sam = ula_sampler_full_jax_jit(key, jax.vmap(model.potential), jax.vmap(jax.jacfwd(model.potential)), n_iter, eps, X0)

# For subset
# sam = ula_sampler_full_jax_jit(key, potential_subset, gradient_subset, n_iter, eps, X0_subset)

# For whole damn thing


sam = ula_sampler_full_jax_jit(key, model.minusLogLikelihood, model.gradient_minusLogLikelihood, n_iter, eps, X0, model.lower_bound, model.upper_bound)

In [ ]:
reshaped_matrix = np.array(sam.reshape((sam.shape[0] * sam.shape[1], 11)))
fig = corner.corner(reshaped_matrix[-50000:], hist_kwargs={'density':True}, labels=model.gwfast_param_order, truths=model.true_params)

In [ ]:
sam = ula_sampler_full_jax_jit(key, model.minusLogLikelihood, model.gradient_minusLogLikelihood, n_iter, eps, sam[-1], model.lower_bound, model.upper_bound)

In [ ]:
# Draw several particles from prior
X = model._newDrawFromPrior(3)

# Francesco derivative
test1 = model.gradient_minusLogLikelihood(X)

# Take gradient of likelihood directly
# f = jax.jit(jax.jacfwd(model.minusLogLikelihood))
# gl = jax.jacobian(model.minusLogLikelihood)

# Check to see if this works
# gl(X[0].squeeze())

# f = jax.vmap(jax.jacfwd(model.minusLogLikelihood))
# f(X)
# model.gradient_minusLogLikelihood(X)

In [ ]:
test1

In [ ]:
np.allclose(test2[np.arange(3), np.arange(3), :], test1)

In [ ]:
test2 = f(X)

In [ ]:
a.shape

In [ ]:
print(model.gwfast_param_order)

In [ ]:
sam.shape

In [ ]:
reshaped_matrix = np.array(sam.reshape((sam.shape[0] * sam.shape[1], 11)))
fig = corner.corner(reshaped_matrix[-50000:], hist_kwargs={'density':True}, labels=model.gwfast_param_order, truths=model.true_params)

In [ ]:
sam = ula_sampler_full_jax_jit(key, potential_subset, gradient_subset, n_iter, eps, sam[-1])

In [ ]:
reshaped_matrix = np.array(sam.reshape((n_iter * n_particles, len(subset_params))))
labels = np.array(model.gwfast_param_order)[subset_params]
fig = corner.corner(reshaped_matrix[-50000:], hist_kwargs={'density':True}, labels=labels, truths=injection[subset_params])

In [ ]:
import corner
reshaped_matrix = sam.reshape((n_iter * n_particles, model.DoF))
reshaped_matrix = np.array(reshaped_matrix)
fig = corner.corner(reshaped_matrix[-20000:], hist_kwargs={'density':True}, truths=jnp.mean(bounded_iid_samples, axis=0)) # For rosenbrock
# fig = corner.corner(np.array(reshaped_matrix[-60000:]), hist_kwargs={'density':True}, truths=jnp.mean(bounded_iid_samples, axis=0)) # For rosenbrock
corner.corner(bounded_iid_samples[-20000:], color='r', fig=fig, hist_kwargs={'density':True})


# GW PROBLEM

In [ ]:

# Center periodic coordinates 
# for param in ['Phicoal', 'psi', 'phi']:
#     x = model.injParams[param][0]
#     delta = x - (model.priorDict[param][1] + model.priorDict[param][0]) / 2
#     model.priorDict[param][0] += delta
#     model.priorDict[param][1] += delta

# model.lower_bound = model.lower_bound.at[4].set(model.priorDict['phi'][0])
# model.upper_bound = model.upper_bound.at[4].set(model.priorDict['phi'][1])

# model.lower_bound = model.lower_bound.at[8].set(model.priorDict['Phicoal'][0])
# model.upper_bound = model.upper_bound.at[8].set(model.priorDict['Phicoal'][1])

# model.lower_bound = model.lower_bound.at[6].set(model.priorDict['psi'][0])
# model.upper_bound = model.upper_bound.at[6].set(model.priorDict['psi'][1])


# gamma = lambda t: 1 # Annealing schedule
# periodic_coordinates = jnp.array([4, 6, 8])
# bounded_coordinates = jnp.array([0, 1, 2, 3, 5, 7, 9, 10])
# standard_dev = jnp.ones(11) * 0.01
# f = jnp.array([1, 2, 1])
# X = X.at[:,4].set(jnp.mod(X[:,4], 2*jnp.pi)) # phi
# X = X.at[:,6].set(jnp.mod(X[:,6], jnp.pi))   # psi
# X = X.at[:,8].set(jnp.mod(X[:,8], 2*jnp.pi)) # phi_c

